# Train the Bibley models (Kaggle)

Settings → Accelerator → **GPU T4 x1** (or P100), Internet **on**. Run top to bottom.

Kaggle gives about 30 free GPU hours a week and 12 hours per session, so train a few models per session.

In [ ]:
!git clone -b bibley-models https://github.com/olivercraftboycompany-design/bibley-ai.git
%cd bibley-ai
!pip install -q -r training/requirements-train.txt

In [ ]:
# Build the KJV training set (about 15,200 examples)
!python training/build_dataset.py

In [ ]:
# Pick which models to train this session (names from training/models.json)
MODELS = [
    'Bibley-Smol-135M',
    'Bibley-Qwen3-0.6B',
    'Bibley-Granite-350M',
]
for m in MODELS:
    !python training/train_lora.py --model {m}

In [ ]:
# How often does each model quote the verse exactly?
for m in MODELS:
    !python training/eval_quotes.py training/out/{m}/merged --n 100

In [ ]:
# Shrink each model under 1 GB as GGUF
for m in MODELS:
    !bash training/export_gguf.sh {m}

In [ ]:
# Optional: upload to your Hugging Face account.
# Add a secret named HF_TOKEN under Add-ons → Secrets first.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
api = HfApi(token=UserSecretsClient().get_secret('HF_TOKEN'))
user = api.whoami()['name']
for m in MODELS:
    repo = f'{user}/{m}'
    api.create_repo(repo, exist_ok=True)
    api.upload_folder(folder_path=f'training/out/{m}/gguf', repo_id=repo)
    print('uploaded', repo)